# YouDub 全流程：视频 → 翻译 → 配音 → 成品视频
选择 GPU 运行时。全部重计算在 Colab，翻译调用 MiniMax-M3。先用 30–90 秒视频验收。


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, sys, os, subprocess, uuid, json
bundle = files.upload()
assert len(bundle) == 1, '请选择 youdub-pipeline-colab.zip'
root = Path('/content/youdub-pipeline')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(bundle.values())))) as archive:
    assert 'scripts/colab_pipeline.py' in archive.namelist(), '不是全流程源码包'
    for info in archive.infolist():
        assert (root/info.filename).resolve().is_relative_to(root.resolve()), 'Invalid ZIP path'
    archive.extractall(root)
del bundle
os.chdir(root)


## 安装（首次需要几分钟）
专用 Python 环境复用 Colab 的 CUDA PyTorch，避免给 notebook 内核安装互相冲突的依赖。


In [ ]:
import importlib.metadata as metadata
torch_version = metadata.version('torch')
audio_version = metadata.version('torchaudio')
minor = '.'.join(torch_version.split('+')[0].split('.')[:2])
codec_versions = {'2.5':'0.1', '2.6':'0.2', '2.7':'0.5', '2.8':'0.7', '2.9':'0.9', '2.10':'0.10', '2.11':'0.11'}
assert minor in codec_versions, f'请先更新 TorchCodec 兼容配置：torch={torch_version}'
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'fonts-noto-cjk', 'fontconfig', 'git'], check=True)
venv = root/'.venv-colab'
# virtualenv seeds its own pip and repairs incomplete environments.
setup = subprocess.run([sys.executable, 'scripts/colab_environment.py', str(root)],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(setup.stdout)
setup.check_returncode()
python = str(venv/'bin/python')
constraints = root/'colab-constraints.txt'
constraints.write_text(f'torch=={torch_version}\ntorchaudio=={audio_version}\ntorchcodec=={codec_versions[minor]}.*\n')
subprocess.run([python, '-m', 'pip', 'install', '-c', str(constraints), '-r', 'requirements-colab.txt'], check=True)
demucs = root/'submodule/demucs'
commit = 'e976d93ecc3865e5757426930257e200846a520a'
if not (demucs/'.git').exists():
    demucs.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'init', str(demucs)], check=True)
subprocess.run(['git', '-C', str(demucs), 'fetch', '--depth', '1', 'https://github.com/facebookresearch/demucs.git', commit], check=True)
subprocess.run(['git', '-C', str(demucs), 'checkout', '--detach', commit], check=True)
print('安装完成；继续运行预检')


In [ ]:
subprocess.run([python, 'scripts/colab_preflight.py'], check=True)


## 翻译配置
在左侧钥匙图标添加 `MINIMAX_API_KEY` 并允许此 notebook 访问；也可以在隐藏输入框粘贴。不会打印或保存密钥到源码包。


In [ ]:
from google.colab import userdata
from getpass import getpass
try:
    key = userdata.get('MINIMAX_API_KEY')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    key = getpass('MiniMax API Key: ')
assert key and key.strip(), 'API Key 不能为空'
run_env = os.environ.copy()
run_env.update(OPENAI_API_KEY=key.strip(), OPENAI_BASE_URL='https://api.minimaxi.com/v1',
    OPENAI_MODEL='MiniMax-M3', OPENAI_TRANSLATE_CONCURRENCY='2', DEVICE='cuda',
    FUNASR_DEVICE='cuda:0', DEMUCS_DEVICE='cuda', DEMUCS_CHUNK_SECONDS='60',
    FUNASR_MODEL='FunAudioLLM/SenseVoiceSmall', FUNASR_HUB='hf', FUNASR_VAD_HUB='ms',
    VOXCPM_MODEL='OpenBMB/VoxCPM2', VOXCPM_LOAD_DENOISER='false',
    VOXCPM_LOW_MEMORY_INIT='true', VOXCPM_OPTIMIZE='false',
    WORKFOLDER=str(root/'workfolder'), MODEL_CACHE_DIR=str(root/'model-cache'))
del key
print('MiniMax-M3 配置就绪（密钥不显示）')


## 上传视频
默认英语 → 中文，输出配音和硬字幕。已有视频在 Colab 时，可直接把 video 设为它的绝对路径，跳过 files.upload()。重新执行本单元会创建新任务。


In [ ]:
direction = 'en-zh' # en-zh / ja-zh / zh-en
output_mode = 'both' # both / subtitles / dubbing
uploaded = files.upload()
assert len(uploaded) == 1, '请选择一个视频'
name, content = next(iter(uploaded.items()))
video = root/('input-' + uuid.uuid4().hex + Path(name).suffix)
video.write_bytes(content)
del uploaded, content
state = root/('task-' + uuid.uuid4().hex + '.json')
result_zip = state.with_suffix('.zip')


## 执行／失败后续跑
再次执行本单元会复用同一任务的成功阶段。运行时回收后，数据库和中间文件会丢失；下载 ZIP 不是完整运行时备份。


In [ ]:
previous_process = globals().get('process')
if previous_process is not None:
    assert previous_process.poll() is not None, '上一个任务仍在运行'
run_env.update(VOXCPM_LOW_MEMORY_INIT='true', VOXCPM_OPTIMIZE='false')
command = [python, '-u', 'scripts/colab_pipeline.py', '--state', str(state), '--result', str(result_zip)]
if state.exists():
    command += ['--resume']
    if previous_process is not None and previous_process.returncode != 0:
        command += ['--recover-dead-pid', str(previous_process.pid)]
else:
    command += ['--video', str(video), '--direction', direction, '--output-mode', output_mode]
process = subprocess.Popen(command, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
assert returncode == 0, f'流水线退出码 {returncode}；负值表示信号终止。保留当前运行时，修复后续跑。'


In [ ]:
assert result_zip.is_file(), '先完成处理单元'
files.download(str(result_zip))
print('ZIP 包含最终视频、识别结果、译文和字幕；请先检查音画同步和中文配音。')
